In [ ]:
import pandas as pd
import requests
import zipfile
import io
import datetime
from sqlalchemy import create_engine, text, inspect
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==============================================================================
# 1. CONFIGURAÇÕES & CONEXÃO
# ==============================================================================

# Carrega as variáveis do arquivo .env
load_dotenv()

# ==============================================================================
# 1. CONFIGURAÇÕES SEGURAS (COM DOTENV)
# ==============================================================================

# Busca as credenciais nas variáveis de ambiente
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Tratamento de caracteres especiais (Isso resolve o problema do @ e do ç)
# O quote_plus transforma 'senha@123' em 'senha%40123' automaticamente
encoded_user = quote_plus(DB_USER)
encoded_pass = quote_plus(DB_PASS)

# Monta a string de conexão segura
DB_CONNECTION = f"postgresql+psycopg2://{encoded_user}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Schema Bronze
SCHEMA_BRONZE = 'layer_01_bronze'
BASE_URL_CVM = 'http://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/FRE/DADOS/'

In [ ]:
# ==============================================================================
# 2. FUNÇÕES DE SUPORTE (ENGINEERING TOOLS)
# ==============================================================================

def criar_engine_db():
    return create_engine(DB_CONNECTION)

def configurar_tabela_logs(engine):
    """
    Cria a tabela de logs dentro do schema layer_01_bronze.
    Essa tabela conta a história da execução.
    """
    query_create_log = f"""
    CREATE TABLE IF NOT EXISTS {SCHEMA_BRONZE}.fre_raw_logs_by_timestamp (
        log_id SERIAL PRIMARY KEY,
        data_execucao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        nivel_log VARCHAR(50),
        ano_referencia INT,
        arquivo_origem VARCHAR(200),
        tabela_destino VARCHAR(200),
        mensagem TEXT,
        schema_drift_detectado BOOLEAN DEFAULT FALSE
    );
    """
    with engine.connect() as conn:
        # Garante que o schema existe
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_BRONZE};"))
        conn.execute(text(query_create_log))
        conn.commit()
    print(f"✅ Setup: Tabela de logs verificada em {SCHEMA_BRONZE}.")

def registrar_log(engine, nivel, ano, arquivo, tabela, msg, drift=False):
    """Insere o registro no banco."""
    query = text(f"""
        INSERT INTO {SCHEMA_BRONZE}.fre_raw_logs_by_timestamp 
        (nivel_log, ano_referencia, arquivo_origem, tabela_destino, mensagem, schema_drift_detectado, data_execucao)
        VALUES (:niv, :ano, :arq, :tab, :msg, :drift, NOW())
    """)
    with engine.connect() as conn:
        conn.execute(query, {'niv': nivel, 'ano': ano, 'arq': arquivo, 'tab': tabela, 'msg': msg, 'drift': drift})
        conn.commit()
    
    # Feedback visual para o aluno
    icon = "🟢" if nivel == 'SUCCESS' else "🟡" if nivel == 'WARNING' else "🔴"
    print(f"{icon} [{nivel}] {msg}")

def verificar_carga_existente(engine, nome_arquivo_zip):
    """Idempotência: Verifica se este ZIP já foi carregado com sucesso."""
    query = text(f"""
        SELECT 1 FROM {SCHEMA_BRONZE}.fre_raw_logs_by_timestamp
        WHERE arquivo_origem = :arq 
          AND nivel_log = 'SUCCESS'
          AND mensagem LIKE 'Processamento do ZIP finalizado%'
    """)
    with engine.connect() as conn:
        return conn.execute(query, {'arq': nome_arquivo_zip}).fetchone() is not None

def verificar_schema_drift(engine, nome_tabela, df_novo):
    """
    🕵️ DETETIVE DE DADOS:
    Compara as colunas do DataFrame novo com as colunas que já existem na tabela do banco.
    Retorna uma mensagem se houver colunas novas (Drift).
    """
    inspector = inspect(engine)
    
    # Se a tabela não existe, não há drift, é a primeira carga (Full Load)
    if not inspector.has_table(nome_tabela, schema=SCHEMA_BRONZE):
        return False, "Tabela Nova (Primeira Carga)"

    # Pega colunas do banco
    colunas_banco = [col['name'] for col in inspector.get_columns(nome_tabela, schema=SCHEMA_BRONZE)]
    colunas_novo_arquivo = df_novo.columns.tolist()

    # Compara conjuntos (Set Difference)
    novas_colunas = set(colunas_novo_arquivo) - set(colunas_banco)

    if novas_colunas:
        return True, f"⚠️ Schema Drift Detectado! Novas colunas encontradas: {novas_colunas}"
    
    return False, "Estrutura mantida (Sem alterações)"

# ==============================================================================
# 3. ETL PRINCIPAL
# ==============================================================================

def run_etl_pipeline():
    engine = criar_engine_db()
    configurar_tabela_logs(engine)
    
    ano_atual = datetime.datetime.now().year
    lista_anos = range(2010, ano_atual + 1)

    # ------------------------------------------------------------------
    # TRUQUE DE ENGENHARIA: HEADERS (User-Agent)
    # Isso faz o Python fingir que é um navegador Chrome para não ser bloqueado
    # ------------------------------------------------------------------
    headers_navegador = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"\n🚀 Iniciando varredura de 2010 até {ano_atual} com identidade de navegador...\n")

    for ano in lista_anos:
        nome_arquivo_zip = f"fre_cia_aberta_{ano}.zip"
        
        # Dica: Use HTTPS para evitar problemas de redirecionamento
        url_download = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/FRE/DADOS/{nome_arquivo_zip}"
        
        # Verificação de Idempotência
        if verificar_carga_existente(engine, nome_arquivo_zip):
            print(f"⏭️  Ano {ano} já processado. Pulando.")
            continue

        # 1. Verificação de Existência (Agora passando os headers!)
        try:
            check_response = requests.head(url_download, headers=headers_navegador, timeout=10)
            
            # Se der erro (403, 404, 500), a gente loga e pula
            if check_response.status_code != 200:
                msg_erro = f"Falha no acesso HTTP. Status Code: {check_response.status_code}"
                print(f"❌ {msg_erro} para o ano {ano}")
                
                # AGORA SIM: Gravamos no log por que falhou
                registrar_log(engine, 'ERROR', ano, nome_arquivo_zip, '-', msg_erro)
                continue
                
        except Exception as e:
            registrar_log(engine, 'ERROR', ano, nome_arquivo_zip, '-', f"Erro de conexão ao checar arquivo: {e}")
            continue

        try:
            # 2. Download Real (Passando headers novamente)
            registrar_log(engine, 'INFO', ano, nome_arquivo_zip, '-', 'Iniciando Download...')
            r = requests.get(url_download, headers=headers_navegador, timeout=60)
            
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                arquivos_csv = [f for f in z.namelist() if f.endswith('.csv')]
                
                if not arquivos_csv:
                    registrar_log(engine, 'WARNING', ano, nome_arquivo_zip, '-', 'Zip vazio.')
                    continue

                for csv_file in arquivos_csv:
                    nome_base = csv_file.lower().replace(f'_{ano}.csv', '').replace('.csv', '')
                    if not nome_base.startswith('fre_'):
                        nome_tabela_final = f"fre_{nome_base}"
                    else:
                        nome_tabela_final = nome_base

                    try:
                        with z.open(csv_file) as f:
                            df = pd.read_csv(f, sep=';', encoding='windows-1252', on_bad_lines='skip', low_memory=False)
                        
                        df['metadata_data_carga'] = datetime.datetime.now()
                        df['metadata_arquivo_origem'] = csv_file
                        df = df.astype(str)

                        # Verificação de Drift
                        is_drift, msg_drift = verificar_schema_drift(engine, nome_tabela_final, df)
                        if is_drift:
                            registrar_log(engine, 'WARNING', ano, csv_file, nome_tabela_final, msg_drift, drift=True)
                        
                        # Carga
                        df.to_sql(
                            nome_tabela_final, 
                            engine, 
                            schema=SCHEMA_BRONZE, 
                            if_exists='append', 
                            index=False,
                            method='multi',
                            chunksize=1000
                        )
                        
                        registrar_log(engine, 'INFO', ano, csv_file, nome_tabela_final, f"Sucesso: {len(df)} linhas.")

                    except Exception as e:
                        registrar_log(engine, 'ERROR', ano, csv_file, nome_tabela_final, f"Erro no CSV: {str(e)}")

            registrar_log(engine, 'SUCCESS', ano, nome_arquivo_zip, 'TODAS', 'Processamento do ZIP finalizado.')

        except Exception as e:
            registrar_log(engine, 'FATAL', ano, nome_arquivo_zip, '-', f"Erro crítico no download: {str(e)}")

In [ ]:
if __name__ == "__main__":
    run_etl_pipeline()